In [3]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
import numpy as np
from tqdm import tqdm

print("Hello World")

ModuleNotFoundError: No module named 'datasets'

In [4]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

True
1
NVIDIA GeForce RTX 3070


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
import numpy as np
from tqdm import tqdm

class DeepfakeDataset(Dataset):
    """Custom PyTorch Dataset for Deepfake classification"""
    
    def __init__(self, hf_dataset, transform=None, preload=False):
        """
        Args:
            hf_dataset: Hugging Face dataset object
            transform: Optional torchvision transforms
            preload: Whether to load all images into memory upfront
        """
        self.dataset = hf_dataset
        self.transform = transform
        self.preload = preload
        
        if preload:
            self.images = []
            self.labels = []
            print("Preloading dataset into memory...")
            for item in tqdm(hf_dataset, desc="Preloading"):
                self.images.append(item['image'])
                self.labels.append(item['label'])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if self.preload:
            image = self.images[idx]
            label = self.labels[idx]
        else:
            item = self.dataset[idx]
            image = item['image']
            label = item['label']
        
        # Convert grayscale to RGB if needed
        if image.mode != 'RGB':
            image = image.convert('RGB')
            
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)  # Using long for classification

def get_transforms(img_size=224):
    """Return train and validation transforms"""
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.1)
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_transform

def load_datasets(batch_size=32, num_workers=4, img_size=224, val_split=0.1):
    """Load and prepare datasets with DataLoaders"""
    # Load HF dataset
    dataset = load_dataset("pujanpaudel/deepfake_face_classification")
    hf_train = dataset['train']
    
    # Get transforms
    train_transform, val_transform = get_transforms(img_size)
    
    # Create full dataset
    full_dataset = DeepfakeDataset(
        hf_dataset=hf_train,
        transform=train_transform,  # Default to train transform
        preload=True  # Preload for faster training
    )
    
    # Split dataset
    val_size = int(val_split * len(full_dataset))
    train_size = len(full_dataset) - val_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
    
    # Apply different transform to validation set
    val_dataset.transform = val_transform
    
    # Create DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True  # Helps with batch norm
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size*2,  # Larger batches for validation
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )
    
    return train_loader, val_loader

if __name__ == "__main__":
    # Example usage
    train_loader, val_loader = load_datasets()
    
    # Inspect sample batch
    sample_imgs, sample_labels = next(iter(train_loader))
    print(f"\n✅ Sample batch shape: {sample_imgs.shape}")
    print(f"✅ Sample labels: {sample_labels[:8]}")
    print(f"✅ Training batches: {len(train_loader)}, Validation batches: {len(val_loader)}")



ResNet50

# Filename: train_resnet_deepfake.py
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.metrics import (accuracy_score, roc_auc_score, 
                           confusion_matrix, classification_report,
                           precision_recall_curve, average_precision_score)
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os
import json
from datetime import datetime
from torchsummary import summary
from data_loader import load_datasets  # Make sure this matches your data loader

# Set device and reproducibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)

def build_resnet50(freeze=True, dropout=0.5, pretrained=True):
    """Build ResNet50 model with customizable head
    
    Args:
        freeze: Whether to freeze base layers
        dropout: Dropout probability
        pretrained: Use pretrained weights
        
    Returns:
        Configured ResNet50 model
    """
    weights = models.ResNet50_Weights.DEFAULT if pretrained else None
    model = models.resnet50(weights=weights)
    
    # Freeze layers if specified
    if freeze:
        for param in model.parameters():
            param.requires_grad = False
    
    # Always unfreeze last layer and optionally layer4
    for name, param in model.named_parameters():
        if name.startswith("layer4") or name.startswith("fc"):
            param.requires_grad = True
    
    # Replace classifier head
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(512, 1),
        nn.Sigmoid()
    )
    
    return model.to(device)

def train_model(model, train_loader, val_loader, experiment_dir, epochs=15, lr=1e-4):
    """Training loop with early stopping and model checkpointing
    
    Args:
        model: Initialized model
        train_loader: Training data loader
        val_loader: Validation data loader  
        experiment_dir: Path to save results
        epochs: Maximum training epochs
        lr: Initial learning rate
        
    Returns:
        Trained model and training history
    """
    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=2, factor=0.5, verbose=True
    )
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'accuracy': [],
        'auc': [],
        'precision': [],
        'recall': [],
        'f1': []
    }
    
    best_metrics = {'auc': 0.0, 'epoch': 0}
    early_stop_counter = 0
    patience = 3
    
    for epoch in range(epochs):
        # Training Phase
        model.train()
        train_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        
        for inputs, labels in progress_bar:
            inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1).float()
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            progress_bar.set_postfix({'loss': loss.item()})
        
        train_loss /= len(train_loader.dataset)
        history['train_loss'].append(train_loss)
        
        # Validation Phase
        model.eval()
        val_loss = 0.0
        y_true, y_pred = [], []
        
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1).float()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(outputs.cpu().numpy())
        
        val_loss /= len(val_loader.dataset)
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        y_pred_class = (y_pred > 0.5).astype(int)
        
        # Calculate metrics
        acc = accuracy_score(y_true, y_pred_class)
        auc = roc_auc_score(y_true, y_pred)
        report = classification_report(y_true, y_pred_class, output_dict=True)
        
        history['val_loss'].append(val_loss)
        history['accuracy'].append(acc)
        history['auc'].append(auc)
        history['precision'].append(report['weighted avg']['precision'])
        history['recall'].append(report['weighted avg']['recall']) 
        history['f1'].append(report['weighted avg']['f1-score'])
        
        # Update scheduler
        scheduler.step(val_loss)
        
        # Print epoch summary
        print(f"\nEpoch {epoch+1}/{epochs} Summary:")
        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"Accuracy: {acc:.4f} | AUC: {auc:.4f}")
        print(f"Precision: {report['weighted avg']['precision']:.4f}")
        print(f"Recall: {report['weighted avg']['recall']:.4f}")
        print(f"F1-Score: {report['weighted avg']['f1-score']:.4f}")
        print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")
        
        # Checkpoint best model
        if auc > best_metrics['auc']:
            best_metrics = {
                'auc': auc,
                'accuracy': acc,
                'epoch': epoch+1,
                'val_loss': val_loss
            }
            early_stop_counter = 0
            torch.save({
                'epoch': epoch+1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': val_loss,
                'auc': auc
            }, os.path.join(experiment_dir, "best_model.pth"))
            print("✅ New best model saved!")
        else:
            early_stop_counter += 1
            if early_stop_counter >= patience:
                print(f"⏹ Early stopping triggered at epoch {epoch+1}")
                break
    
    # Save training history
    with open(os.path.join(experiment_dir, "training_history.json"), 'w') as f:
        json.dump(history, f)
    
    # Save best metrics
    with open(os.path.join(experiment_dir, "best_metrics.json"), 'w') as f:
        json.dump(best_metrics, f)
    
    return model, history

def plot_training(history, experiment_dir):
    """Plot training metrics and save to experiment directory"""
    plt.figure(figsize=(18, 12))
    
    # Loss plot
    plt.subplot(2, 3, 1)
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'], label='Validation')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Accuracy plot
    plt.subplot(2, 3, 2)
    plt.plot(history['accuracy'])
    plt.title('Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    
    # AUC plot
    plt.subplot(2, 3, 3)
    plt.plot(history['auc'])
    plt.title('Validation AUC')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    
    # Precision-Recall plot
    plt.subplot(2, 3, 4)
    plt.plot(history['precision'], label='Precision')
    plt.plot(history['recall'], label='Recall')
    plt.title('Precision and Recall')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.legend()
    
    # F1-Score plot
    plt.subplot(2, 3, 5)
    plt.plot(history['f1'])
    plt.title('F1-Score')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    
    plt.tight_layout()
    plt.savefig(os.path.join(experiment_dir, "training_metrics.png"), dpi=300)
    plt.close()

def evaluate_model(model, loader, experiment_dir):
    """Evaluate model performance and save results"""
    model.eval()
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc="Evaluating"):
            inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1).float()
            outputs = model(inputs)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(outputs.cpu().numpy())
    
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    y_pred_class = (y_pred > 0.5).astype(int)
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred_class),
        'auc': roc_auc_score(y_true, y_pred),
        'average_precision': average_precision_score(y_true, y_pred),
        'confusion_matrix': confusion_matrix(y_true, y_pred_class).tolist()
    }
    
    # Classification report
    report = classification_report(
        y_true, y_pred_class, 
        target_names=["Real", "Fake"],
        output_dict=True
    )
    
    # Save metrics
    with open(os.path.join(experiment_dir, "evaluation_metrics.json"), 'w') as f:
        json.dump({**metrics, **report}, f)
    
    # Plot confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(metrics['confusion_matrix'], annot=True, fmt='d', 
                cmap='Blues', xticklabels=["Real", "Fake"], 
                yticklabels=["Real", "Fake"])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.savefig(os.path.join(experiment_dir, "confusion_matrix.png"), dpi=300)
    plt.close()
    
    # Plot PR curve
    precision, recall, _ = precision_recall_curve(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, label=f'AP={metrics["average_precision"]:.2f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend()
    plt.savefig(os.path.join(experiment_dir, "pr_curve.png"), dpi=300)
    plt.close()
    
    # Print summary
    print("\n📊 Evaluation Results:")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"AUC: {metrics['auc']:.4f}")
    print(f"Average Precision: {metrics['average_precision']:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred_class, target_names=["Real", "Fake"]))
    
    return metrics

if __name__ == "__main__":
    # Setup experiment
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    experiment_dir = os.path.join("experiments", f"resnet50_{timestamp}")
    os.makedirs(experiment_dir, exist_ok=True)
    
    # Load data
    print("🚀 Loading datasets...")
    train_loader, val_loader = load_datasets()
    
    # Initialize model
    print("🛠 Building model...")
    model = build_resnet50(freeze=True, dropout=0.5)
    summary(model, input_size=(3, 224, 224), device=device.type)
    
    # Train model
    print("🏋️ Starting training...")
    model, history = train_model(
        model, 
        train_loader, 
        val_loader, 
        experiment_dir,
        epochs=15,
        lr=1e-4
    )
    
    # Save final model
    torch.save(model.state_dict(), os.path.join(experiment_dir, "final_model.pth"))
    print("💾 Final model saved")
    
    # Plot training history
    plot_training(history, experiment_dir)
    
    # Evaluate on validation set
    print("\n🔍 Evaluating on validation set...")
    val_metrics = evaluate_model(model, val_loader, experiment_dir)
    
    print(f"\n🎉 Experiment complete! Results saved to: {experiment_dir}")



EfficientNetB0

import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from typing import Optional, List, Dict, Tuple
from collections import OrderedDict
import warnings

class EfficientNetB0Binary(nn.Module):
    """Enhanced EfficientNetB0 for binary classification with comprehensive monitoring"""
    
    def __init__(self,
                 freeze_backbone: bool = True,
                 dropout_rate: float = 0.4,
                 pretrained: bool = True,
                 unfreeze_layers: Optional[List[str]] = None,
                 custom_head: Optional[nn.Module] = None,
                 feature_dim: int = 256,
                 gradient_checkpointing: bool = False):
        """
        Initialize EfficientNetB0 model with enhanced capabilities.
        
        Args:
            freeze_backbone: Freeze feature extractor if True
            dropout_rate: Dropout probability (0.0-1.0)
            pretrained: Use ImageNet pretrained weights
            unfreeze_layers: List of layer patterns to unfreeze
            custom_head: Custom classifier head module
            feature_dim: Dimension of intermediate features
            gradient_checkpointing: Enable memory-efficient training
        """
        super().__init__()
        
        # Validate inputs
        if not 0 <= dropout_rate <= 1:
            raise ValueError("dropout_rate must be between 0 and 1")
            
        if feature_dim <= 0:
            raise ValueError("feature_dim must be positive")
        
        # Load pretrained weights with validation
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.base_model = efficientnet_b0(weights=weights)
        
        # Enable gradient checkpointing if requested
        if gradient_checkpointing:
            self.base_model.features.gradient_checkpointing = True
            warnings.warn("Gradient checkpointing enabled - tradeoff between memory and speed")
        
        # Configuration tracking
        self.config = {
            'freeze_backbone': freeze_backbone,
            'dropout_rate': dropout_rate,
            'pretrained': pretrained,
            'unfreeze_layers': unfreeze_layers,
            'feature_dim': feature_dim,
            'gradient_checkpointing': gradient_checkpointing,
            'input_size': (3, 224, 224)  # Standard EfficientNetB0 input
        }
        
        # Setup layers
        self._configure_layers(freeze_backbone, unfreeze_layers)
        in_features = self.base_model.classifier[1].in_features
        self.base_model.classifier = self._build_head(
            in_features, feature_dim, dropout_rate, custom_head)
        
        # Device setup
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(self.device)
        
        # Activation monitoring
        self.activations = OrderedDict()
        self.gradients = OrderedDict()
        self._register_hooks()

    def _configure_layers(self, freeze_backbone: bool, unfreeze_layers: Optional[List[str]]):
        """Configure layer freezing with validation"""
        if freeze_backbone:
            for param in self.base_model.parameters():
                param.requires_grad = False
        
        # Default layers to unfreeze (last block + classifier)
        if unfreeze_layers is None:
            unfreeze_layers = ["features.6", "classifier"]
        
        # Validate unfreeze patterns
        valid_layers = set(name for name, _ in self.base_model.named_parameters())
        for pattern in unfreeze_layers:
            if not any(pattern in name for name in valid_layers):
                warnings.warn(f"Unfreeze pattern '{pattern}' doesn't match any layers")
        
        # Apply unfreezing
        for name, param in self.base_model.named_parameters():
            if any(pattern in name for pattern in unfreeze_layers):
                param.requires_grad = True

    def _build_head(self, 
                   in_features: int,
                   feature_dim: int,
                   dropout_rate: float,
                   custom_head: Optional[nn.Module]) -> nn.Module:
        """Build classifier head with feature extraction capability"""
        if custom_head is not None:
            if not isinstance(custom_head, nn.Module):
                raise TypeError("custom_head must be a nn.Module")
            return custom_head
        
        return nn.Sequential(
            OrderedDict([
                ('dropout1', nn.Dropout(p=dropout_rate)),
                ('linear1', nn.Linear(in_features, feature_dim)),
                ('bn1', nn.BatchNorm1d(feature_dim)),
                ('silu', nn.SiLU(inplace=True)),
                ('dropout2', nn.Dropout(p=dropout_rate/2)),
                ('linear2', nn.Linear(feature_dim, 1)),
                ('sigmoid', nn.Sigmoid())
            ])
        )

    def _register_hooks(self):
        """Register forward and backward hooks to monitor layer activations and gradients"""
        def get_activation_hook(name):
            def hook(module, input, output):
                self.activations[name] = output.detach()
            return hook
        
        def get_gradient_hook(name):
            def hook(module, grad_input, grad_output):
                self.gradients[name] = grad_output[0].detach()
            return hook
        
        # Monitor key layers
        self.handles = []
        for name, layer in self.base_model.named_children():
            if name in ['features.6', 'classifier']:
                # Forward hook
                self.handles.append(
                    layer.register_forward_hook(get_activation_hook(name))
                # Backward hook
                self.handles.append(
                    layer.register_full_backward_hook(get_gradient_hook(name)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass with activation tracking"""
        if x.shape[1:] != torch.Size(self.config['input_size']):
            warnings.warn(f"Input size {x.shape[1:]} doesn't match expected {self.config['input_size']}")
        
        self.activations.clear()
        self.gradients.clear()
        return self.base_model(x)

    def get_feature_extractor(self) -> nn.Module:
        """Return feature extractor with avgpool"""
        return nn.Sequential(
            self.base_model.features,
            self.base_model.avgpool
        )

    def get_features(self) -> Dict[str, torch.Tensor]:
        """Get intermediate features from last forward pass"""
        return self.activations

    def get_gradients(self) -> Dict[str, torch.Tensor]:
        """Get gradients from last backward pass"""
        return self.gradients

    def get_trainable_params(self) -> List[str]:
        """Get names of trainable parameters"""
        return [name for name, param in self.named_parameters() 
                if param.requires_grad]

    def get_param_counts(self) -> Tuple[int, int]:
        """Get counts of trainable and total parameters"""
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return trainable, total

    def get_config(self) -> Dict:
        """Get model configuration"""
        return self.config

    def __del__(self):
        """Cleanup hooks when model is deleted"""
        for handle in self.handles:
            handle.remove()

def build_efficientnetb0(
    freeze: bool = True,
    dropout: float = 0.4,
    pretrained: bool = True,
    unfreeze_layers: Optional[List[str]] = None,
    custom_head: Optional[nn.Module] = None,
    feature_dim: int = 256,
    gradient_checkpointing: bool = False
) -> EfficientNetB0Binary:
    """
    Build configured EfficientNetB0Binary with enhanced options.
    
    Args:
        freeze: Freeze base layers
        dropout: Dropout rate (0.0-1.0)
        pretrained: Use pretrained weights
        unfreeze_layers: Layer patterns to unfreeze
        custom_head: Custom classifier
        feature_dim: Intermediate feature dimension
        gradient_checkpointing: Enable memory-efficient training
        
    Returns:
        Configured EfficientNetB0Binary
    """
    return EfficientNetB0Binary(
        freeze_backbone=freeze,
        dropout_rate=dropout,
        pretrained=pretrained,
        unfreeze_layers=unfreeze_layers,
        custom_head=custom_head,
        feature_dim=feature_dim,
        gradient_checkpointing=gradient_checkpointing
    )

Densenet121


from densenet121_binary import build_densenet121
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from sklearn.metrics import (accuracy_score, roc_auc_score, 
                           classification_report, confusion_matrix,
                           precision_recall_curve, average_precision_score)
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime
from tqdm import tqdm

# ------------------ Configuration ------------------ #
class Config:
    def __init__(self):
        self.data_dir = "faceforensics_dataset_binary"
        self.batch_size = 32
        self.img_size = 224
        self.val_split = 0.1
        self.epochs = 15
        self.lr = 1e-4
        self.freeze = True
        self.dropout = 0.4
        self.patience = 3
        
        # Create experiment directory
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.exp_dir = os.path.join("experiments", f"densenet121_{timestamp}")
        os.makedirs(self.exp_dir, exist_ok=True)
        self.model_path = os.path.join(self.exp_dir, "best_model.pth")
        
    def save(self):
        """Save configuration to experiment directory"""
        config_dict = {k:v for k,v in vars(self).items() if not k.startswith('__')}
        with open(os.path.join(self.exp_dir, "config.json"), 'w') as f:
            json.dump(config_dict, f, indent=2)

# ------------------ Enhanced Data Loader ------------------ #
def prepare_dataloaders(cfg):
    """Prepare train and validation dataloaders with augmentation"""
    train_transform = transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # Load dataset
    full_dataset = ImageFolder(root=cfg.data_dir)
    
    # Split dataset
    val_len = int(len(full_dataset) * cfg.val_split)
    train_len = len(full_dataset) - val_len
    train_set, val_set = random_split(full_dataset, [train_len, val_len])
    
    # Apply separate transforms
    train_set.dataset.transform = train_transform
    val_set.dataset.transform = val_transform
    
    # Create dataloaders
    train_loader = DataLoader(
        train_set, 
        batch_size=cfg.batch_size, 
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_set, 
        batch_size=cfg.batch_size*2,  # Larger batches for validation
        num_workers=4,
        pin_memory=True
    )
    
    # Save class names
    with open(os.path.join(cfg.exp_dir, "class_names.json"), 'w') as f:
        json.dump(full_dataset.classes, f)
    
    return train_loader, val_loader

# ------------------ Training with Early Stopping ------------------ #
def train_model(model, train_loader, val_loader, cfg):
    """Training loop with early stopping and learning rate scheduling"""
    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=2, factor=0.5)
    criterion = nn.BCELoss()
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'accuracy': [],
        'auc': [],
        'lr': []
    }
    
    best_auc = 0.0
    early_stop_counter = 0
    
    for epoch in range(cfg.epochs):
        # Training phase
        model.train()
        epoch_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs} [Train]")
        
        for x, y in progress_bar:
            x, y = x.to(model.device), y.float().unsqueeze(1).to(model.device)
            
            optimizer.zero_grad()
            preds = model(x)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * x.size(0)
            progress_bar.set_postfix({'loss': loss.item()})
        
        train_loss = epoch_loss / len(train_loader.dataset)
        history['train_loss'].append(train_loss)
        
        # Validation phase
        val_loss, y_true, y_scores = evaluate_model(model, val_loader, criterion)
        history['val_loss'].append(val_loss)
        
        # Calculate metrics
        y_pred_class = (torch.tensor(y_scores) > 0.5).int()
        acc = accuracy_score(y_true, y_pred_class)
        auc = roc_auc_score(y_true, y_scores)
        
        history['accuracy'].append(acc)
        history['auc'].append(auc)
        history['lr'].append(optimizer.param_groups[0]['lr'])
        
        # Update scheduler
        scheduler.step(auc)
        
        # Print epoch summary
        print(f"\nEpoch {epoch+1}/{cfg.epochs} Summary:")
        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"Val Accuracy: {acc:.4f} | Val AUC: {auc:.4f}")
        print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")
        
        # Checkpoint best model
        if auc > best_auc:
            best_auc = auc
            early_stop_counter = 0
            torch.save({
                'epoch': epoch+1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_auc': auc,
                'val_loss': val_loss,
            }, cfg.model_path)
            print("✅ New best model saved!")
        else:
            early_stop_counter += 1
            if early_stop_counter >= cfg.patience:
                print(f"⏹ Early stopping triggered at epoch {epoch+1}")
                break
    
    # Save training history
    with open(os.path.join(cfg.exp_dir, "training_history.json"), 'w') as f:
        json.dump(history, f)
    
    return history

# ------------------ Enhanced Evaluation ------------------ #
def evaluate_model(model, loader, criterion=None):
    """Evaluate model performance and return metrics"""
    model.eval()
    y_true, y_scores = [], []
    total_loss = 0.0
    
    with torch.no_grad():
        for x, y in tqdm(loader, desc="Evaluating"):
            x, y = x.to(model.device), y.float().unsqueeze(1).to(model.device)
            preds = model(x)
            
            if criterion:
                loss = criterion(preds, y)
                total_loss += loss.item() * x.size(0)
            
            y_true.extend(y.cpu().numpy())
            y_scores.extend(preds.cpu().numpy())
    
    metrics = {
        'y_true': np.array(y_true),
        'y_scores': np.array(y_scores),
        'y_pred': (np.array(y_scores) > 0.5).astype(int)
    }
    
    if criterion:
        metrics['loss'] = total_loss / len(loader.dataset)
    
    return metrics

def generate_evaluation_report(model, loader, cfg):
    """Generate comprehensive evaluation report with visualizations"""
    metrics = evaluate_model(model, loader)
    
    # Calculate metrics
    report = {
        'accuracy': accuracy_score(metrics['y_true'], metrics['y_pred']),
        'auc': roc_auc_score(metrics['y_true'], metrics['y_scores']),
        'ap': average_precision_score(metrics['y_true'], metrics['y_scores']),
        'confusion_matrix': confusion_matrix(metrics['y_true'], metrics['y_pred']).tolist(),
        'classification_report': classification_report(
            metrics['y_true'], metrics['y_pred'], 
            target_names=["Real", "Fake"],
            output_dict=True
        )
    }
    
    # Save metrics
    with open(os.path.join(cfg.exp_dir, "evaluation_report.json"), 'w') as f:
        json.dump(report, f, indent=2)
    
    # Plot confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(report['confusion_matrix'], annot=True, fmt='d', 
                cmap='Blues', xticklabels=["Real", "Fake"], 
                yticklabels=["Real", "Fake"])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.savefig(os.path.join(cfg.exp_dir, "confusion_matrix.png"), dpi=300)
    plt.close()
    
    # Plot PR curve
    precision, recall, _ = precision_recall_curve(metrics['y_true'], metrics['y_scores'])
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, label=f'AP={report["ap"]:.2f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend()
    plt.savefig(os.path.join(cfg.exp_dir, "pr_curve.png"), dpi=300)
    plt.close()
    
    # Print summary
    print("\n📊 Evaluation Results:")
    print(f"Accuracy: {report['accuracy']:.4f}")
    print(f"AUC: {report['auc']:.4f}")
    print(f"Average Precision: {report['ap']:.4f}")
    print("\nClassification Report:")
    print(classification_report(
        metrics['y_true'], metrics['y_pred'], 
        target_names=["Real", "Fake"])
    )
    
    return report

# ------------------ Visualization ------------------ #
def plot_training_history(history, cfg):
    """Plot training metrics and save to experiment directory"""
    plt.figure(figsize=(15, 10))
    
    # Loss plot
    plt.subplot(2, 2, 1)
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'], label='Validation')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Accuracy plot
    plt.subplot(2, 2, 2)
    plt.plot(history['accuracy'])
    plt.title('Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    
    # AUC plot
    plt.subplot(2, 2, 3)
    plt.plot(history['auc'])
    plt.title('Validation AUC')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    
    # Learning rate plot
    plt.subplot(2, 2, 4)
    plt.plot(history['lr'])
    plt.title('Learning Rate')
    plt.xlabel('Epoch')
    plt.ylabel('LR')
    plt.yscale('log')
    
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.exp_dir, "training_metrics.png"), dpi=300)
    plt.close()

# ------------------ Main ------------------ #
if __name__ == "__main__":
    # Initialize configuration
    cfg = Config()
    cfg.save()
    
    print(f"🚀 Starting experiment in {cfg.exp_dir}")
    
    # Prepare data
    print("📂 Loading data...")
    train_loader, val_loader = prepare_dataloaders(cfg)
    
    # Build model
    print("🧠 Building model...")
    model = build_densenet121(freeze=cfg.freeze, dropout=cfg.dropout)
    print(f"💻 Using device: {model.device}")
    
    # Train model
    print("🏋️ Training model...")
    history = train_model(model, train_loader, val_loader, cfg)
    
    # Plot training history
    plot_training_history(history, cfg)
    
    # Load best model for evaluation
    print("\n🔍 Loading best model for evaluation...")
    checkpoint = torch.load(cfg.model_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Generate evaluation report
    generate_evaluation_report(model, val_loader, cfg)
    
    print(f"\n🎉 Experiment complete! Results saved to: {cfg.exp_dir}")


MobileNetV3

from mobilenetv3_binary import build_mobilenetv3
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from torchvision.datasets import ImageFolder
from sklearn.metrics import (accuracy_score, roc_auc_score, 
                            classification_report, confusion_matrix,
                            precision_recall_curve, average_precision_score,
                            f1_score, recall_score, precision_score)
import os, json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime
from tqdm import tqdm
import pandas as pd
from torch.utils.tensorboard import SummaryWriter
import warnings
from collections import defaultdict
from torch.cuda.amp import GradScaler, autocast

# Suppress unnecessary warnings
warnings.filterwarnings('ignore', category=UserWarning)

# ------------------ Enhanced Configuration ------------------ #
class Config:
    def __init__(self):
        # Data configuration
        self.data_dir = "faceforensics_dataset_binary"
        self.batch_size = 64  # Optimized for modern GPUs
        self.img_size = 256   # Better resolution for detection
        self.val_split = 0.15 # More reliable validation
        self.test_split = 0.15
        self.class_weights = True  # Handle class imbalance
        
        # Training configuration
        self.epochs = 30
        self.lr = 3e-4       # Optimized learning rate
        self.min_lr = 1e-6     # Minimum learning rate
        self.weight_decay = 1e-4  # Better regularization
        self.freeze_backbone = True
        self.freeze_epochs = 5  # Freeze initial layers
        self.dropout = 0.4     # Better regularization
        
        # Augmentation configuration
        self.augment = True
        self.color_jitter = 0.3
        self.random_erase_prob = 0.2
        
        # Optimization configuration
        self.optimizer = "AdamW"  # Best for this task
        self.scheduler = "CosineAnnealingWarmRestarts"  # Better learning rate adaptation
        self.patience = 5
        self.early_stop = True
        self.mixed_precision = True  # Faster training
        
        # Model checkpointing
        self.save_top_k = 3  # Save multiple checkpoints
        
        # Experiment tracking
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.exp_name = f"mobilenetv3_{timestamp}"
        self.exp_dir = os.path.join("experiments", self.exp_name)
        os.makedirs(self.exp_dir, exist_ok=True)
        
        # Path configurations
        self.model_path = os.path.join(self.exp_dir, "best_model.pth")
        self.last_model_path = os.path.join(self.exp_dir, "last_model.pth")
        self.log_dir = os.path.join(self.exp_dir, "logs")
        
    def save(self):
        with open(os.path.join(self.exp_dir, "config.json"), 'w') as f:
            json.dump(vars(self), f, indent=2)
            
    def __str__(self):
        return json.dumps(vars(self), indent=2)

# ------------------ Optimized Data Loader ------------------ #
def prepare_dataloaders(cfg):
    # Enhanced augmentations
    train_transform = transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(
            brightness=cfg.color_jitter,
            contrast=cfg.color_jitter,
            saturation=cfg.color_jitter,
            hue=0.1
        ),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=cfg.random_erase_prob, scale=(0.02, 0.2)), 
    ])
    
    # Validation/test transforms
    val_transform = transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # Load dataset
    dataset = ImageFolder(cfg.data_dir)
    
    # Save class names
    with open(os.path.join(cfg.exp_dir, "class_names.json"), 'w') as f:
        json.dump(dataset.classes, f)
    
    # Handle class imbalance
    if cfg.class_weights:
        class_counts = np.bincount([y for _, y in dataset])
        class_weights = 1. / class_counts
        class_weights = class_weights / class_weights.sum()
        sample_weights = [class_weights[y] for _, y in dataset]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
        shuffle = False
    else:
        sampler = None
        shuffle = True
    
    # Split dataset
    val_len = int(len(dataset) * cfg.val_split)
    test_len = int(len(dataset) * cfg.test_split)
    train_len = len(dataset) - val_len - test_len
    
    train_set, val_set, test_set = random_split(dataset, [train_len, val_len, test_len])
    
    # Apply transforms
    train_set.dataset.transform = train_transform
    val_set.dataset.transform = val_transform
    test_set.dataset.transform = val_transform
    
    # Create optimized dataloaders
    train_loader = DataLoader(
        train_set, 
        batch_size=cfg.batch_size, 
        shuffle=shuffle, 
        sampler=sampler,
        num_workers=8, 
        pin_memory=True,
        persistent_workers=True
    )
    
    val_loader = DataLoader(
        val_set, 
        batch_size=cfg.batch_size * 2, 
        num_workers=4, 
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_set,
        batch_size=cfg.batch_size * 2,
        num_workers=4,
        pin_memory=True
    )
    
    return train_loader, val_loader, test_loader

# ------------------ Optimized Evaluation ------------------ #
def evaluate_model(model, loader, criterion=None, device=None):
    if device is None:
        device = next(model.parameters()).device
        
    model.eval()
    y_true, y_scores, y_pred = [], [], []
    total_loss = 0.0
    
    with torch.no_grad():
        for x, y in tqdm(loader, desc="Evaluating", leave=False):
            x, y = x.to(device), y.float().unsqueeze(1).to(device)
            
            with autocast():
                preds = model(x)
                if criterion:
                    loss = criterion(preds, y)
                    total_loss += loss.item() * x.size(0)
            
            y_true.extend(y.cpu().numpy())
            y_scores.extend(torch.sigmoid(preds).cpu().numpy())
            y_pred.extend((preds > 0).int().cpu().numpy())
    
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)
    y_pred = np.array(y_pred)
    
    metrics = {
        'loss': total_loss / len(loader.dataset) if criterion else 0.0,
        'accuracy': accuracy_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_scores),
        'f1': f1_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'ap': average_precision_score(y_true, y_scores)
    }
    
    return metrics, y_true, y_scores, y_pred

# ------------------ Optimized Training Loop ------------------ #
def train_model(model, train_loader, val_loader, cfg):
    # Setup device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    # Setup optimization
    if cfg.optimizer == "AdamW":
        optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    else:
        optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    
    # Setup learning rate scheduler
    if cfg.scheduler == "ReduceLROnPlateau":
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=2, min_lr=cfg.min_lr
        )
    else:
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=10, T_mult=1, eta_min=cfg.min_lr
        )
    
    # Loss function
    criterion = nn.BCEWithLogitsLoss()
    
    # Mixed precision training
    scaler = GradScaler(enabled=cfg.mixed_precision)
    
    # Initialize tracking
    history = defaultdict(list)
    best_metrics = {'auc': 0.0}
    early_stop_counter = 0
    writer = SummaryWriter(cfg.log_dir)
    
    # Training loop
    for epoch in range(cfg.epochs):
        # Unfreeze backbone after certain epochs
        if cfg.freeze_backbone and epoch == cfg.freeze_epochs:
            print("\nUnfreezing backbone layers...")
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        
        # Training phase
        model.train()
        running_loss = 0.0
        train_metrics = defaultdict(float)
        
        for batch_idx, (x, y) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs}")):
            x, y = x.to(device), y.float().unsqueeze(1).to(device)
            
            optimizer.zero_grad(set_to_none=True)
            
            # Mixed precision forward
            with autocast(enabled=cfg.mixed_precision):
                preds = model(x)
                loss = criterion(preds, y)
            
            # Backward pass
            scaler.scale(loss).backward()
            
            # Gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            # Update weights
            scaler.step(optimizer)
            scaler.update()
            
            # Update metrics
            running_loss += loss.item() * x.size(0)
            y_pred = (preds > 0).int()
            train_metrics['acc'] += accuracy_score(y.cpu().numpy(), y_pred.cpu().numpy()) * x.size(0)
        
        # Calculate epoch metrics
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = train_metrics['acc'] / len(train_loader.dataset)
        
        # Validation phase
        val_metrics, _, _, _ = evaluate_model(model, val_loader, criterion, device)
        
        # Learning rate scheduling
        if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_metrics['auc'])
        else:
            scheduler.step()
        
        # Update history
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_auc'].append(val_metrics['auc'])
        history['val_f1'].append(val_metrics['f1'])
        history['lr'].append(optimizer.param_groups[0]['lr'])
        
        # TensorBoard logging
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_metrics['loss'], epoch)
        writer.add_scalar('Accuracy/train', train_acc, epoch)
        writer.add_scalar('Accuracy/val', val_metrics['accuracy'], epoch)
        writer.add_scalar('AUC/val', val_metrics['auc'], epoch)
        writer.add_scalar('LR', optimizer.param_groups[0]['lr'], epoch)
        
        # Print epoch summary
        print(f"\nEpoch {epoch+1}/{cfg.epochs}:")
        print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['accuracy']:.4f}")
        print(f"  AUC: {val_metrics['auc']:.4f} | F1: {val_metrics['f1']:.4f}")
        print(f"  LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        # Save best model
        if val_metrics['auc'] > best_metrics['auc']:
            best_metrics = val_metrics
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_metrics': val_metrics
            }, cfg.model_path)
            print("✅ New best model saved!")
            early_stop_counter = 0
        else:
            early_stop_counter += 1
        
        # Early stopping
        if cfg.early_stop and early_stop_counter >= cfg.patience:
            print(f"⏹ Early stopping triggered at epoch {epoch+1}")
            break

    # Save final model
    torch.save({
        'epoch': cfg.epochs,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_metrics': val_metrics
    }, cfg.last_model_path)
    
    # Save training history
    with open(os.path.join(cfg.exp_dir, "training_history.json"), 'w') as f:
        json.dump(history, f, indent=2)
    
    writer.close()
    return history, best_metrics

# ------------------ Enhanced Evaluation Report ------------------ #
def generate_evaluation_report(model, loader, cfg, phase="val"):
    metrics, y_true, y_scores, y_pred = evaluate_model(model, loader, device=next(model.parameters()).device)
    
    report = {
        'metrics': metrics,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'classification_report': classification_report(
            y_true, y_pred, target_names=["Real", "Fake"], output_dict=True
        )
    }
    
    # Save report
    report_path = os.path.join(cfg.exp_dir, f"{phase}_report.json")
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    
    # Plot confusion matrix
    plt.figure(figsize=(6, 6))
    sns.heatmap(report['confusion_matrix'], annot=True, fmt='d', cmap='Blues', 
                xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"])
    plt.title(f"Confusion Matrix ({phase.capitalize()})")
    plt.savefig(os.path.join(cfg.exp_dir, f"{phase}_confusion_matrix.png"))
    plt.close()
    
    # Plot PR curve
    precision, recall, _ = precision_recall_curve(y_true, y_scores)
    plt.figure(figsize=(6, 4))
    plt.plot(recall, precision, label=f"AP = {metrics['ap']:.2f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve ({phase.capitalize()})")
    plt.legend()
    plt.savefig(os.path.join(cfg.exp_dir, f"{phase}_pr_curve.png"))
    plt.close()
    
    # Print summary
    print(f"\n📊 {phase.capitalize()} Evaluation:")
    print(f"Accuracy: {metrics['accuracy']:.4f} | AUC: {metrics['auc']:.4f} | F1: {metrics['f1']:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=["Real", "Fake"]))
    
    return report

# ------------------ Enhanced Training Visualization ------------------ #
def plot_training_history(history, cfg):
    plt.figure(figsize=(18, 12))
    
    # Plot loss
    plt.subplot(2, 2, 1)
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'], label='Validation')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Plot accuracy
    plt.subplot(2, 2, 2)
    plt.plot(history['train_acc'], label='Train')
    plt.plot(history['val_acc'], label='Validation')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    # Plot AUC
    plt.subplot(2, 2, 3)
    plt.plot(history['val_auc'], label='Validation')
    plt.title('Validation AUC')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    
    # Plot learning rate
    plt.subplot(2, 2, 4)
    plt.plot(history['lr'])
    plt.title('Learning Rate')
    plt.xlabel('Epoch')
    plt.ylabel('LR')
    
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.exp_dir, "training_metrics.png"))
    plt.close()

# ------------------ Main Execution ------------------ #
if __name__ == "__main__":
    # Initialize configuration
    cfg = Config()
    cfg.save()
    print(f"\n⚙️ Configuration:\n{cfg}")
    
    # Prepare data
    print("\n📂 Loading and preparing data...")
    train_loader, val_loader, test_loader = prepare_dataloaders(cfg)
    
    # Build model
    print("\n🧠 Building model...")
    model = build_mobilenetv3(freeze=cfg.freeze_backbone, dropout=cfg.dropout)
    print(f"Model architecture:\n{model}")
    
    # Train model
    print("\n🏋️ Starting training...")
    history, best_metrics = train_model(model, train_loader, val_loader, cfg)
    
    # Plot training history
    plot_training_history(history, cfg)
    
    # Evaluate best model
    print("\n🔍 Evaluating best model on validation set...")
    checkpoint = torch.load(cfg.model_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    val_report = generate_evaluation_report(model, val_loader, cfg, "val")
    
    # Evaluate on test set
    print("\n🧪 Evaluating on test set...")
    test_report = generate_evaluation_report(model, test_loader, cfg, "test")
    
    print(f"\n🎉 Training complete! Best validation AUC: {best_metrics['auc']:.4f}")
    print(f"📁 Results saved in: {cfg.exp_dir}")